In [ ]:
import sciline as sl
from sciline.reporter import TimingReporter
import scippnexus as snx
from ess import bifrost
import ess.bifrost.data
from ess.reduce import nexus
from ess.reduce.nexus.types import *
from ess.reduce.nexus import open_component_group, load_all_data
from ess.spectroscopy.types import *
import scipp as sc
from ess.spectroscopy.indirect.ki import primary_spectrometer_coordinate_transformation_graph
from ess.spectroscopy.indirect.kf import secondary_spectrometer_coordinate_transformation_graph

In [ ]:
from ess.bifrost.io.nexus import _get_inputs
from ess.spectroscopy.indirect.conversion import add_spectrometer_coords

def load_bifrost_analyzer_componentss(
    detector_locations: NeXusAllComponentLocationSpec[snx.NXdetector, RunType],
    detector_components: AllNeXusComponents[snx.NXdetector, RunType],
) -> AllNeXusComponents[snx.NXcrystal, RunType]:
    out = {}
    instrument_location = NeXusLocationSpec(
        filename=detector_locations.filename,
        entry_name=detector_locations.entry_name,
        component_name=None,
        selection=detector_locations.selection,
    )
    with open_component_group(instrument_location, nx_class=snx.NXinstrument) as instrument_group:
        for detector in detector_components.values():
            detector_name = detector['nexus_component_name']
            analyzer = _find_class_in_inputs(instrument_group, snx.NXcrystal, instrument_group[detector_name])[()]
            analyzer['detector_name'] = detector_name
            out[detector_name] = analyzer
    return AllNeXusComponents[snx.NXcrystal, RunType](sc.DataGroup(out))


def _find_class_in_inputs(
    base: snx.Group,
    target: type,
    start: snx.Group,
) -> snx.Group:
    pending = [start]
    while pending:
        element = pending.pop(0)  # BFS to avoid going down long chains that do not lead to an analyzer
        if element.nx_class == target:
            return element

        match _get_inputs(element):
            case [inp]:
                try:
                    pending.append(base[inp])
                except KeyError as error:  # noqa: PERF203
                    error.add_note(f"The input NeXus component {inp} of {element.name} does not exist.")
                    raise
            case inputs:
                raise RuntimeError(f"Expected a single NeXus input component in {element.name}. Got {len(inputs)}")
    raise ValueError(f"No NXcrystal found in the inputs of detector {detector_name}")


def get_calibrated_analyzers(
    analyzer_components: AllNeXusComponents[snx.NXcrystal, RunType],
    interval: TimeInterval[RunType],
    time_filter: TransformationTimeFilter[
        snx.NXcrystal, RunType
    ],
) -> Analyzers[RunType]:
    return Analyzers[RunType](sc.DataGroup({
        analyzer_name: get_calibrated_analyzer(
            component,
            nexus.workflow.to_transformation(
                chain=component['depends_on'],
                interval=interval,
                time_filter=time_filter,
            )
        )
        for analyzer_name, component in analyzer_components.items()    
    }))

        
def get_calibrated_analyzer(
    analyzer_component: NeXusComponent[snx.NXcrystal, RunType],
    analyzer_transform: NeXusTransformation[snx.NXcrystal, RunType],
) -> Analyzer[RunType]:
    return Analyzer[RunType](
        sc.DataGroup(
            dspacing=analyzer_component['d_spacing'],
            position=nexus.workflow.compute_dynamic_position(analyzer_transform),
            transform=analyzer_transform,
            detector_name=analyzer_component['detector_name'],
        )
    )


def get_calibrated_detector_bifrost(
    detector_components: AllNeXusComponents[snx.NXdetector, RunType],
    analyzers: Analyzers[RunType],
    *,
    primary_graph: PrimarySpecCoordTransformGraph[RunType],
    secondary_graph: SecondarySpecCoordTransformGraph[RunType],
    interval: TimeInterval[RunType],
    time_filter: TransformationTimeFilter[
        snx.NXcrystal, RunType
    ],
) -> EmptyDetector[RunType]:
    from ess.bifrost.detector import get_base_calibrated_detector_bifrost, arc_and_channel_from_detector_number

    analyzer_parameters = []
    detectors = []
    for analyzer in analyzers.values():
        detector = detector_components[analyzer['detector_name']]
        transform=nexus.workflow.to_transformation(
            chain=detector['depends_on'],
            interval=interval,
            time_filter=time_filter,
        )
        da = get_base_calibrated_detector_bifrost(detector, transform=transform)
        da = da.rename(dim_0='tube', dim_1='length')
        
        arc, channel = arc_and_channel_from_detector_number(da.coords['detector_number'])
        da.coords['arc'] = arc
        da.coords['channel'] = channel
        
        detectors.append(da)
        analyzer_parameters.append(_extract_analyzer_parameters(da, analyzer))

    detector = sc.concat(detectors, dim='bank')

    
    ap = {key: [val] for key, val in analyzer_parameters[0].items()}
    for a in analyzer_parameters[1:]:
        for key, val in a.items():
            ap[key].append(val)
    apg = {key: _just(sc.concat(vals, dim='bank')) for key, vals in ap.items()}
    detector = add_spectrometer_coords(
            detector,
            primary_graph,
            SecondarySpecCoordTransformGraph[RunType](
                {**secondary_graph, **apg}
            ),
        )
    return detector


# can't use lambda in dict comprehension because it would bind to dynamic `vals`
def _just(value):
    def impl():
        return value
    return impl

    
def _extract_analyzer_parameters(
    detector: sc.DataArray,
    analyzer: Analyzer[RunType],
) -> dict[str, sc.Variable]:
    ana_pos: Position[snx.NXcrystal, RunType] = analyzer['position']
    if ana_pos.is_dynamic:
        if 'time' not in detector.coords:
            raise sc.CoordError(
                "The analyzer position is time-dependent but the detector is not"
            )
        analyzer_positions = ana_pos.positions
        if not sc.identical(analyzer_positions.coords['time'], detector.coords['time']):
            raise sc.CoordError(
                f"The analyzer and detector positions are not at the same times.\n"
                f"Analyzer: {analyzer_positions.coords['time']}\n"
                f"Detector: {detector.coords['time']}\n"
                "This is likely due to a change in the NeXus structure. It used to "
                "guarantee that the times are identical."
            )
        analyzer_position = analyzer_positions.data
        analyzer_transform = analyzer['transform'].value.data
    else:
        analyzer_position = ana_pos.position
        analyzer_transform = analyzer['transform'].value

    return {
        'analyzer_dspacing': analyzer['dspacing'],
        'analyzer_position': analyzer_position,
        'analyzer_transform': analyzer_transform,
    }


def load_bifrost_detector_data(
    locations: NeXusAllDataLocationSpec[snx.NXdetector, RunType],
) -> NeXusData[snx.NXdetector, RunType]:
    # TODO We need to associate data with banks so that assemble_detector_data
    #      can put it all together. But the events loaded here lack the required
    #      info. We could either
    #      1. Use more bespoke code than `load_all_data` and also load the detector number.
    #         - This would duplicate the logic in `get_calibrated_detector_bifrost` and
    #           require access to more low-level functions of ess.reduce.
    #      2. Deduce arc number and channel from the group name. (data group key)
    #         - Not very intrusive
    #         - But can we rely on the naming scheme in the file? (If so, we need an extra nightly test).
    data = load_all_data(
        file_path=locations.filename,
        selection=locations.selection,
        nx_class=snx.NXdetector,
        entry_name=locations.entry_name,
    )
    return data

In [ ]:
# TODO allow limiting detector names

providers = [
    bifrost.nexus.moderator_class_for_source,
    load_bifrost_analyzer_componentss,  # override load_all_nexus_components
    primary_spectrometer_coordinate_transformation_graph,
    secondary_spectrometer_coordinate_transformation_graph,
    get_calibrated_analyzers,
    get_calibrated_detector_bifrost,
    load_bifrost_detector_data,
]
params = {
    Filename[SampleRun]: bifrost.data.simulated_elastic_incoherent_with_phonon(),
    **bifrost.io.nexus.parameters,
}

wf = nexus.workflow.GenericNeXusWorkflow(run_types=[SampleRun], monitor_types=[NormalizationMonitor])
for provider in providers: wf.insert(provider)
for key, param in params.items(): wf[key] = param

In [ ]:
wf.visualize(NeXusData[snx.NXdetector, SampleRun], graph_attr={'rankdir':'LR'})

In [ ]:
data = wf.compute(NeXusData[snx.NXdetector, SampleRun])

In [ ]:
timer = TimingReporter()
res = wf.compute([EmptyDetector[SampleRun]], reporter=timer, scheduler=sl.scheduler.NaiveScheduler())

In [ ]:
print(timer.summary())

In [ ]:
res[EmptyDetector[SampleRun]]

In [ ]:
c = wf.compute(AllNeXusComponents[snx.NXdetector, SampleRun])

In [ ]:
c['channel_1_1_triplet']